# 📝 Object-Oriented Programming
### Exercises & Solutions — 30 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Classes, `__init__`, instance vs class attributes (1-3)
- Encapsulation: properties, getters/setters, private/protected (4-7)
- Inheritance: single, multiple, MRO, `super()` (8-12)
- Polymorphism & duck typing (13-15)
- Abstraction: ABCs, interfaces (16-18)
- Dunder/magic methods: `__eq__`, `__lt__`, `__add__`, `__len__`, `__getitem__`, `__iter__`, `__call__`, `__repr__`, `__hash__`, `__enter__/__exit__` (19-26)
- Composition, mixins, design patterns (Factory, Observer, Strategy, Singleton) (27-30)


---


### 1. Basic Class with Instance + Class Attributes

Create a `Car` class with a class attribute `wheels = 4` (shared) and instance attributes `make`, `model`, `year`. Add a method `info()` returning a formatted string.

In [ ]:
class Car:
    wheels = 4
    def __init__(self, make, model, year):
        self.make, self.model, self.year = make, model, year
    def info(self):
        return f"{self.year} {self.make} {self.model} ({self.wheels} wheels)"

c1 = Car("Toyota", "Corolla", 2022)
c2 = Car("Tesla", "Model 3", 2023)
print(c1.info())
print(c2.info())
print("Shared class attr:", Car.wheels, c1.wheels is c2.__class__.wheels)

### 2. Class Attribute Mutation Pitfall

Demonstrate the difference between mutating a class attribute via the class vs an instance — show how `instance.attr = x` creates a NEW instance attribute shadowing the class one.

In [ ]:
class Counter:
    total = 0          # class attribute
    def __init__(self):
        Counter.total += 1   # mutate via the CLASS

c1, c2, c3 = Counter(), Counter(), Counter()
print("Shared total:", Counter.total)   # 3

c1.total = 99   # this creates an INSTANCE attribute, doesn't touch the class one!
print("c1.total:", c1.total)             # 99 (instance attr)
print("c2.total:", c2.total)             # 3  (still reads class attr)
print("Counter.total:", Counter.total)   # 3  (unaffected)

### 3. Class Methods as Alternative Constructors

Build a `Date` class with `__init__(self, y, m, d)` and TWO `@classmethod` factories: `from_string(cls, s)` (parses "YYYY-MM-DD") and `today(cls)` (uses the real current date).

In [ ]:
import datetime

class Date:
    def __init__(self, y, m, d):
        self.y, self.m, self.d = y, m, d
    @classmethod
    def from_string(cls, s):
        y, m, d = map(int, s.split("-"))
        return cls(y, m, d)
    @classmethod
    def today(cls):
        t = datetime.date.today()
        return cls(t.year, t.month, t.day)
    def __repr__(self):
        return f"Date({self.y}-{self.m:02d}-{self.d:02d})"

print(Date.from_string("2024-03-15"))
print(Date.today())

### 4. Encapsulation with @property validation

Create a `Temperature` class storing celsius internally, exposing a `celsius` property that rejects values below -273.15, and a computed `fahrenheit` property (read AND write).

In [ ]:
class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius     # goes through the setter, validating immediately

    @property
    def celsius(self):
        return self._celsius
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32
    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5/9     # reuses validation!

t = Temperature(25)
print(t.celsius, t.fahrenheit)
t.fahrenheit = 32
print(t.celsius)
try:
    t.celsius = -300
except ValueError as e:
    print("Blocked:", e)

### 5. Name Mangling & Private Attributes

Show how `__balance` (double underscore) gets name-mangled, and why `_balance` (single underscore) is just a convention, not enforcement.

In [ ]:
class Account:
    def __init__(self, balance):
        self._balance = balance        # protected (convention only)
        self.__secret_pin = "1234"     # private (name-mangled)

acc = Account(100)
print("Convention-protected (still accessible):", acc._balance)

try:
    print(acc.__secret_pin)
except AttributeError as e:
    print("Direct access blocked:", e)

# Name mangling means it's actually stored as _Account__secret_pin
print("Via mangled name:", acc._Account__secret_pin)
print("All attrs:", [a for a in vars(acc)])

### 6. Read-Only Property (no setter)

Create a `Circle` class where `radius` is settable but `area` and `circumference` are READ-ONLY computed properties (no setter defined — attempting to set raises AttributeError).

In [ ]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius
    @property
    def area(self):
        return math.pi * self.radius ** 2
    @property
    def circumference(self):
        return 2 * math.pi * self.radius

c = Circle(5)
print(f"area={c.area:.2f}, circumference={c.circumference:.2f}")
try:
    c.area = 100
except AttributeError as e:
    print("Blocked:", e)

### 7. Validated Setter with Multiple Constraints

Build a `Product` class where `price` must be > 0 and `discount_pct` must be between 0-100; both enforced via property setters, and `final_price` is computed.

In [ ]:
class Product:
    def __init__(self, name, price, discount_pct=0):
        self.name = name
        self.price = price
        self.discount_pct = discount_pct

    @property
    def price(self): return self._price
    @price.setter
    def price(self, v):
        if v <= 0: raise ValueError("price must be > 0")
        self._price = v

    @property
    def discount_pct(self): return self._discount_pct
    @discount_pct.setter
    def discount_pct(self, v):
        if not 0 <= v <= 100: raise ValueError("discount_pct must be 0-100")
        self._discount_pct = v

    @property
    def final_price(self):
        return round(self._price * (1 - self._discount_pct / 100), 2)

p = Product("Laptop", 1000, 15)
print(p.final_price)
for bad in [lambda: Product("X", -5), lambda: Product("X", 10, 150)]:
    try: bad()
    except ValueError as e: print("Blocked:", e)

### 8. Single Inheritance with super()

Build `Animal` (base, has `name`, `make_sound()`) and `Dog(Animal)` that overrides `make_sound()` but calls `super().make_sound()` first to log a generic message.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def make_sound(self):
        return f"{self.name} makes a sound"

class Dog(Animal):
    def make_sound(self):
        base_msg = super().make_sound()
        return f"{base_msg}, specifically: Woof!"

d = Dog("Rex")
print(d.make_sound())

### 9. Multi-level Inheritance Chain

Build a 3-level chain: `Vehicle` → `Car` → `ElectricCar`, each adding an attribute and extending a `describe()` method via `super()`.

In [ ]:
class Vehicle:
    def __init__(self, brand):
        self.brand = brand
    def describe(self):
        return f"Vehicle by {self.brand}"

class Car(Vehicle):
    def __init__(self, brand, doors):
        super().__init__(brand)
        self.doors = doors
    def describe(self):
        return f"{super().describe()}, {self.doors}-door car"

class ElectricCar(Car):
    def __init__(self, brand, doors, battery_kwh):
        super().__init__(brand, doors)
        self.battery_kwh = battery_kwh
    def describe(self):
        return f"{super().describe()}, {self.battery_kwh}kWh battery"

ec = ElectricCar("Tesla", 4, 75)
print(ec.describe())

### 10. Multiple Inheritance & MRO Inspection

Build classes `A`, `B(A)`, `C(A)`, `D(B, C)` each overriding a method `greet()` calling `super().greet()`. Print `D.__mro__` and call `D().greet()` to see the C3 linearization in action.

In [ ]:
class A:
    def greet(self): return "A"
class B(A):
    def greet(self): return f"B->{super().greet()}"
class C(A):
    def greet(self): return f"C->{super().greet()}"
class D(B, C):
    def greet(self): return f"D->{super().greet()}"

print("MRO:", [cls.__name__ for cls in D.__mro__])
print("greet():", D().greet())
# Notice: D -> B -> C -> A -> object, so B's super() goes to C, not directly to A!

### 11. Overriding __init__ Safely with **kwargs

Build a `Shape` base accepting `**kwargs` for extensibility, and a `ColoredShape(Shape)` subclass adding a `color` kwarg while passing the rest up via `super().__init__(**kwargs)`.

In [ ]:
class Shape:
    def __init__(self, name, **kwargs):
        self.name = name
        for k, v in kwargs.items():
            setattr(self, k, v)

class ColoredShape(Shape):
    def __init__(self, name, color, **kwargs):
        super().__init__(name, **kwargs)
        self.color = color

cs = ColoredShape("Square", "red", sides=4, filled=True)
print(vars(cs))

### 12. Diamond Problem Resolved Correctly

Demonstrate that Python's MRO prevents the classic diamond problem: build `Base`, `Left(Base)`, `Right(Base)`, `Bottom(Left, Right)`, and show `Base.__init__` runs only ONCE even though both parents call `super().__init__()`.

In [ ]:
class Base:
    def __init__(self):
        print("Base init")
        self.value = 0

class Left(Base):
    def __init__(self):
        super().__init__()
        print("Left init")

class Right(Base):
    def __init__(self):
        super().__init__()
        print("Right init")

class Bottom(Left, Right):
    def __init__(self):
        super().__init__()
        print("Bottom init")

b = Bottom()
print("MRO:", [c.__name__ for c in Bottom.__mro__])
print("Base.__init__ ran only once thanks to MRO cooperative super() chaining")

### 13. Polymorphism via Common Interface

Build 3 unrelated classes (`Duck`, `Robot`, `Person`) each with a `speak()` method, then write a function that calls `.speak()` on ANY of them without checking type (duck typing).

In [ ]:
class Duck:
    def speak(self): return "Quack!"
class Robot:
    def speak(self): return "Beep boop."
class Person:
    def speak(self): return "Hello!"

def make_it_speak(thing):
    return thing.speak()      # works for ANY object with .speak(), no inheritance needed

for entity in [Duck(), Robot(), Person()]:
    print(f"{type(entity).__name__}: {make_it_speak(entity)}")

### 14. Method Overriding Changes Behaviour Polymorphically

Build a list of mixed `Shape` subclasses (`Square`, `Circle`, `Triangle`) and compute total area with ONE loop calling the same `.area()` method name on each — despite different formulas.

In [ ]:
import math
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...

class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Triangle(Shape):
    def __init__(self, base, height): self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height

shapes = [Square(4), Circle(3), Triangle(5, 6)]
total = sum(s.area() for s in shapes)     # polymorphic dispatch
print(f"Total area: {total:.2f}")
for s in shapes:
    print(f"  {type(s).__name__}: {s.area():.2f}")

### 15. isinstance vs type() for Polymorphic Checks

Show why `isinstance()` should be preferred over `type() ==` when checking types polymorphically, using a small class hierarchy.

In [ ]:
class Animal: pass
class Dog(Animal): pass

d = Dog()
print("type(d) == Animal:", type(d) == Animal)         # False! misses subclasses
print("isinstance(d, Animal):", isinstance(d, Animal))  # True - respects inheritance
print("isinstance(d, Dog):", isinstance(d, Dog))        # True
print("isinstance(d, (Dog, str)):", isinstance(d, (Dog, str)))  # tuple of types also works

### 16. Abstract Base Class Enforces Contract

Build an ABC `PaymentMethod` with abstract `pay(amount)`. Show that instantiating it directly raises `TypeError`, but a concrete subclass implementing `pay()` works.

In [ ]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount): ...

class CreditCard(PaymentMethod):
    def pay(self, amount):
        return f"Charged ${amount} to credit card"

try:
    PaymentMethod()
except TypeError as e:
    print("Blocked:", e)

print(CreditCard().pay(50))

### 17. ABC with Concrete Helper Methods

Build an ABC `Report` with one abstract method `generate_data()` and one CONCRETE method `render()` that calls `generate_data()` internally — demonstrating the Template Method pattern.

In [ ]:
from abc import ABC, abstractmethod

class Report(ABC):
    @abstractmethod
    def generate_data(self): ...

    def render(self):                      # concrete — uses the abstract method
        data = self.generate_data()
        lines = [f"  - {k}: {v}" for k, v in data.items()]
        return f"=== {type(self).__name__} ===\n" + "\n".join(lines)

class SalesReport(Report):
    def generate_data(self):
        return {"total_sales": 50000, "units": 320}

class InventoryReport(Report):
    def generate_data(self):
        return {"in_stock": 1200, "low_stock_items": 5}

for report in [SalesReport(), InventoryReport()]:
    print(report.render())

### 18. Protocol-style Duck Typing (no inheritance needed)

Without using ABC, write a function `total_area(shapes)` that works with ANY object exposing an `.area()` method — demonstrating that Python doesn't require formal interfaces.

In [ ]:
class Box:           # doesn't inherit from anything special
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

class Pizza:          # also unrelated
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

def total_area(shapes):
    return sum(s.area() for s in shapes)    # no type checking at all - pure duck typing

print(total_area([Box(2, 3), Pizza(5), Box(1, 1)]))

### 19. __eq__ and __hash__ for Value Equality

Build a `Point` class with `__eq__` (value equality) and `__hash__` (so it works correctly as a dict key / in a set).

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        if not isinstance(other, Point): return NotImplemented
        return (self.x, self.y) == (other.x, other.y)
    def __hash__(self):
        return hash((self.x, self.y))
    def __repr__(self):
        return f"Point({self.x},{self.y})"

p1, p2, p3 = Point(1,2), Point(1,2), Point(3,4)
print(p1 == p2, p1 == p3)
print(len({p1, p2, p3}))     # set dedups p1/p2 since equal AND same hash
d = {p1: "first"}
print(d[p2])                  # works because p2 hashes/equals same as p1

### 20. __lt__ and total_ordering for Sortable Objects

Build a `Card` class (rank, suit) implementing `__lt__` and use `functools.total_ordering` to get `<=`, `>`, `>=` for free; sort a list of cards.

In [ ]:
from functools import total_ordering

@total_ordering
class Card:
    RANKS = ["2","3","4","5","6","7","8","9","10","J","Q","K","A"]
    def __init__(self, rank, suit):
        self.rank, self.suit = rank, suit
    def __eq__(self, other):
        return self.RANKS.index(self.rank) == self.RANKS.index(other.rank)
    def __lt__(self, other):
        return self.RANKS.index(self.rank) < self.RANKS.index(other.rank)
    def __repr__(self):
        return f"{self.rank}{self.suit}"

cards = [Card("K","♠"), Card("2","♥"), Card("A","♦"), Card("10","♣")]
print(sorted(cards))
print(cards[1] < cards[0], cards[2] > cards[0])  # >= derived automatically!

### 21. __add__, __sub__, __mul__ Operator Overloading

Build a `Vector2D` class supporting `+`, `-`, and scalar `*`, plus `__repr__` for clean display.

In [ ]:
class Vector2D:
    def __init__(self, x, y): self.x, self.y = x, y
    def __add__(self, other): return Vector2D(self.x+other.x, self.y+other.y)
    def __sub__(self, other): return Vector2D(self.x-other.x, self.y-other.y)
    def __mul__(self, scalar): return Vector2D(self.x*scalar, self.y*scalar)
    def __rmul__(self, scalar): return self.__mul__(scalar)   # supports scalar*vector too
    def __repr__(self): return f"Vector2D({self.x}, {self.y})"

v1, v2 = Vector2D(1,2), Vector2D(3,4)
print(v1 + v2, v1 - v2, v1 * 3, 3 * v1)

### 22. __len__ and __contains__ for Container-like Classes

Build a `Playlist` class wrapping an internal list, implementing `__len__` and `__contains__` so `len(playlist)` and `"song" in playlist` work naturally.

In [ ]:
class Playlist:
    def __init__(self, songs=None):
        self.songs = songs or []
    def add(self, song):
        self.songs.append(song)
    def __len__(self):
        return len(self.songs)
    def __contains__(self, song):
        return song in self.songs

pl = Playlist(["Song A", "Song B"])
pl.add("Song C")
print(len(pl))
print("Song B" in pl, "Song Z" in pl)

### 23. __getitem__ and __setitem__ for Indexable Objects

Build a `Matrix` class supporting `matrix[i, j]` get/set syntax using `__getitem__`/`__setitem__` with tuple indices.

In [ ]:
class Matrix:
    def __init__(self, rows, cols):
        self.rows, self.cols = rows, cols
        self.data = [[0]*cols for _ in range(rows)]
    def __getitem__(self, idx):
        i, j = idx
        return self.data[i][j]
    def __setitem__(self, idx, value):
        i, j = idx
        self.data[i][j] = value
    def __repr__(self):
        return "\n".join(str(row) for row in self.data)

m = Matrix(3, 3)
m[0, 0] = 1
m[1, 1] = 1
m[2, 2] = 1
print(m)
print("m[1,1] =", m[1, 1])

### 24. __iter__ and __next__ for Custom Iteration

Build a `Fibonacci` class that's iterable up to n terms, implementing the full iterator protocol (`__iter__` returns self, `__next__` advances state).

In [ ]:
class Fibonacci:
    def __init__(self, limit):
        self.limit = limit
    def __iter__(self):
        self.a, self.b, self.count = 0, 1, 0
        return self
    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = Fibonacci(8)
print(list(fib))
print(list(fib))   # re-iterable! __iter__ resets state each time

### 25. __call__ for Callable Objects

Build a `Multiplier` class whose instances are callable directly (`m(5)`), implementing simple stateful function-like behaviour plus a call counter.

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
        self.call_count = 0
    def __call__(self, x):
        self.call_count += 1
        return x * self.factor

double = Multiplier(2)
triple = Multiplier(3)
print(double(5), triple(5), double(10))
print(f"double was called {double.call_count} times")

### 26. __enter__/__exit__ Custom Context Manager Class

Build a `DatabaseConnection` class-based context manager that prints connect/disconnect messages and suppresses a specific exception type.

In [ ]:
class DatabaseConnection:
    def __enter__(self):
        print("Connecting to DB...")
        self.connected = True
        return self
    def query(self, sql):
        return f"Executing: {sql}"
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Disconnecting from DB...")
        if exc_type is ValueError:
            print(f"Suppressed ValueError: {exc_val}")
            return True     # suppress only ValueError
        return False         # let everything else propagate

with DatabaseConnection() as db:
    print(db.query("SELECT * FROM users"))
    raise ValueError("simulated bad query")
print("Continued after suppressed error")

### 27. Composition: 'has-a' over Inheritance

Build a `Computer` class composed of separate `CPU`, `RAM`, `Storage` objects (has-a relationships), with a `specs()` method aggregating all of them.

In [ ]:
class CPU:
    def __init__(self, model, cores): self.model, self.cores = model, cores
    def __str__(self): return f"{self.model} ({self.cores} cores)"
class RAM:
    def __init__(self, gb): self.gb = gb
    def __str__(self): return f"{self.gb}GB RAM"
class Storage:
    def __init__(self, gb, kind): self.gb, self.kind = gb, kind
    def __str__(self): return f"{self.gb}GB {self.kind}"

class Computer:
    def __init__(self, cpu, ram, storage):
        self.cpu, self.ram, self.storage = cpu, ram, storage   # composed, not inherited
    def specs(self):
        return f"{self.cpu} | {self.ram} | {self.storage}"

pc = Computer(CPU("Ryzen 9", 16), RAM(32), Storage(1000, "SSD"))
print(pc.specs())

### 28. Mixin Classes for Reusable Behaviour

Build a `JSONMixin` and `ComparableMixin` that can be combined into ANY class via multiple inheritance, adding `.to_json()` and comparison ops without a shared base class hierarchy.

In [ ]:
import json

class JSONMixin:
    def to_json(self):
        return json.dumps(self.__dict__)

class ComparableMixin:
    def __eq__(self, other):
        return self.__dict__ == other.__dict__
    def __lt__(self, other):
        return tuple(self.__dict__.values()) < tuple(other.__dict__.values())

class User(JSONMixin, ComparableMixin):
    def __init__(self, name, age):
        self.name, self.age = name, age

u1, u2 = User("Alice", 30), User("Bob", 25)
print(u1.to_json())
print(u1 == User("Alice", 30), u1 < u2 if u1.name < u2.name else u2 < u1)

### 29. Factory Pattern

Build a `ShapeFactory` with a `create(shape_type, **kwargs)` classmethod that instantiates the right `Shape` subclass based on a string key — decoupling construction from usage.

In [ ]:
class Shape:
    def area(self): raise NotImplementedError
class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2
class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

class ShapeFactory:
    _registry = {"square": Square, "circle": Circle}
    @classmethod
    def create(cls, shape_type, **kwargs):
        if shape_type not in cls._registry:
            raise ValueError(f"Unknown shape: {shape_type}")
        return cls._registry[shape_type](**kwargs)

s1 = ShapeFactory.create("square", side=4)
s2 = ShapeFactory.create("circle", r=3)
print(s1.area(), s2.area())

### 30. Observer + Singleton Patterns Combined

Build a `Singleton` `EventBus` (only one instance ever exists) implementing the Observer pattern — multiple subscribers react to published events.

In [ ]:
class EventBus:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._subscribers = []
        return cls._instance

    def subscribe(self, fn):
        self._subscribers.append(fn)
    def publish(self, event):
        for fn in self._subscribers:
            fn(event)

bus1 = EventBus()
bus2 = EventBus()
print("Same instance (singleton):", bus1 is bus2)

bus1.subscribe(lambda e: print(f"Logger received: {e}"))
bus2.subscribe(lambda e: print(f"Alerter received: {e}"))   # adds to the SAME list
bus1.publish("user_signed_up")    # both subscribers fire, since bus1 IS bus2